In [2]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.documents import Document

In [3]:
# Step 1: Sample documents
docs = [
    Document(page_content="LangChain helps build LLM applications."),
    Document(page_content="Pinecone is a vector database for semantic search."),
    Document(page_content="The Eiffel Tower is located in Paris."),
    Document(page_content="Langchain can be used to develop agentic ai application."),
    Document(page_content="Langchain has many types of retrievers.")
]

# Step 2: Create vector store and BM25 retriever
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
dense_vectorstore = FAISS.from_documents(docs, embedding_model)
dense_retriever = dense_vectorstore.as_retriever()

In [5]:
## Sparse retriever
sparse_retriever = BM25Retriever.from_documents(docs)
sparse_retriever.k = 3

# Step 3: Create ensemble retriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.7, 0.3]
)

In [6]:
hybrid_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002B05F00C680>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000002B036E60650>, k=3)], weights=[0.7, 0.3])

In [7]:
# Step 5: Query and get results
query = "How can I build an application using LLMs?"
results = hybrid_retriever.invoke(query)

# Step 6: Print results
for i, doc in enumerate(results):
    print(f"\n🔹 Document {i+1}:\n{doc.page_content}")


🔹 Document 1:
LangChain helps build LLM applications.

🔹 Document 2:
Langchain can be used to develop agentic ai application.

🔹 Document 3:
Langchain has many types of retrievers.

🔹 Document 4:
Pinecone is a vector database for semantic search.


### RAG Pipeline with hybrid retriever

In [17]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain.chat_models.base import init_chat_model

import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GPT-4-MINI_API_KEY"] = os.getenv("GPT-4-MINI_API_KEY")

In [18]:
# Step 5: Prompt Template
prompt = PromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question: {input}
""")

## step 6-llm
llm = init_chat_model(
    model="openai:gpt-4.1-mini",
    api_key=os.getenv("GPT-4-MINI_API_KEY"),
    temperature=0.2,
    base_url="https://models.inference.ai.azure.com"
)
llm

ChatOpenAI(profile={'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000002B0DDA91E50>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002B0DDA9FF50>, root_client=<openai.OpenAI object at 0x000002B0DDA9E8D0>, root_async_client=<openai.AsyncOpenAI object at 0x000002B0DEDBB8C0>, model_name='gpt-4.1-mini', temperature=0.2, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://models.inference.ai.azure.com')

In [19]:
### Create StuffDocumentsChain
combine_docs_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)

## Create full RAG chain
rag_chain = create_retrieval_chain(
    retriever=hybrid_retriever,
    combine_docs_chain=combine_docs_chain
)

rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002B05F00C680>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000002B036E60650>, k=3)], weights=[0.7, 0.3]), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the question based on the context below.\n\nContext:\n{context}\n\nQuestion: {input}\n')
            | ChatOpenAI(profile

In [20]:
# Step 9: Ask a question
query = {"input": "How can I build an app using LLMs?"}
response = rag_chain.invoke(query)

# Step 10: Output
print("Answer:\n", response["answer"])

print("\nSource Documents:")
for i, doc in enumerate(response["context"]):
    print(f"\nDoc {i+1}: {doc.page_content}")

Answer:
 To build an app using LLMs (Large Language Models), you can use LangChain, which is a framework designed to help develop LLM applications. LangChain supports creating agentic AI applications and offers various types of retrievers to efficiently handle information retrieval.

Here’s a high-level approach:

1. **Choose your LLM**: Select a large language model to power your app (e.g., OpenAI's GPT models).

2. **Use LangChain**: Utilize LangChain to integrate the LLM into your application. LangChain provides tools to manage prompts, chains, agents, and retrievers.

3. **Implement Retrieval**: If your app requires semantic search or knowledge retrieval, use LangChain’s retrievers. For scalable and efficient vector search, you can integrate a vector database like Pinecone.

4. **Build Agentic Features**: If your app needs to perform tasks autonomously or interact with external tools, leverage LangChain’s agentic AI capabilities.

5. **Develop and Deploy**: Combine these components